# Structural time-series models

This notebook introduces the direct-use time-series classes that complement the YAML pipeline: reduced-form VAR, Blanchard--Quah long-run identification, proxy SVAR, sign restrictions, local projections, LP-IV, and decomposition utilities. It uses synthetic data so every cell is self-contained.

These are demonstrations of API usage. The separate MATLAB comparator documents the bounded Blanchard--Quah numerical check.

## Contents

1. [Create reproducible data](#Create-reproducible-data)
2. [Fit a reduced-form VAR](#Fit-a-reduced-form-VAR)
3. [Compare structural identification approaches](#Compare-structural-identification-approaches)
4. [Estimate local projections](#Estimate-local-projections)
5. [Decompose VAR dynamics](#Decompose-VAR-dynamics)

In [2]:
import numpy as np
import pandas as pd

from stats_transformer.models.timeseries.identification.blanchard_quah import BlanchardQuahModel
from stats_transformer.models.timeseries.decompositions import TimeSeriesDecompositions
from stats_transformer.models.timeseries.reduced_form.local_projections import LocalProjectionsModel
from stats_transformer.models.timeseries.reduced_form.local_projections_iv import LocalProjectionsIVModel
from stats_transformer.models.timeseries.identification.proxy_svar import ProxySVARModel
# from stats_transformer.models.timeseries.sign_restrictions import SignRestrictionsSVARModel this library is not yet implemented
from stats_transformer.models.timeseries.reduced_form.var import VARModel

## Create reproducible data

In [3]:
rng = np.random.default_rng(42)
observations = 160
shocks = rng.normal(size=(observations, 2))
values = np.zeros((observations, 2))
for index in range(1, observations):
    values[index] = np.array([[0.65, 0.10], [0.20, 0.45]]) @ values[index - 1] + shocks[index]

data = pd.DataFrame(values, columns=["output_growth", "inflation"])
data["instrument"] = shocks[:, 0] + rng.normal(scale=0.25, size=observations)
data["date"] = pd.date_range("1980-01-01", periods=observations, freq="QE")
data.head()

,output_growth,inflation,instrument,date
0,0.000000,0.000000,0.500305,1980-03-31
1,0.750451,0.940565,0.702788,1980-06-30
2,-1.369185,-0.728835,-1.658223,1980-09-30
3,-0.835014,-0.918055,0.315558,1980-12-31
4,-0.651366,-1.433172,0.438360,1981-03-31


## Fit a reduced-form VAR

In [4]:
var_model = VARModel(target_variables=["output_growth", "inflation"], date_column="date", maxlags=2)
var_metrics = var_model.fit(data)
var_metrics

{'aic': np.float64(-0.2191659054684748),
 'bic': np.float64(-0.025330776795882026),
 'hqic': np.float64(-0.1404470188593089),
 'fpe': np.float64(0.8032224135475499),
 'num_observations': 158}

## Compare structural identification approaches

Each approach starts from a reduced-form VAR and imposes a different identifying assumption. Interpret the output only after choosing an economically justified ordering, instrument, or restriction pattern.

In [5]:
bq_model = BlanchardQuahModel(target_variables=["output_growth", "inflation"], date_column="date", maxlags=2)
bq_model.fit(data)
pd.DataFrame(bq_model.B_0, index=["output_growth", "inflation"], columns=["shock_1", "shock_2"])

/Users/minesakikodai/miniforge3/envs/stats-transformer/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)


,shock_1,shock_2
output_growth,0.956060,0.033593
inflation,0.013274,0.909128


In [6]:
proxy_model = ProxySVARModel(target_variables=["output_growth", "inflation"], instrument_variable="instrument", date_column="date", maxlags=2)
proxy_model.fit(data)
proxy_model.impact_column

/Users/minesakikodai/miniforge3/envs/stats-transformer/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)


array([0.95664969, 0.0290032 ])

In [7]:
np.random.seed(42)
sign_model = SignRestrictionsSVARModel(target_variables=["output_growth", "inflation"], sign_pattern=[1, -1], date_column="date", maxlags=2, max_draws=500)
sign_metrics = sign_model.fit(data)
sign_metrics

NameError: name 'SignRestrictionsSVARModel' is not defined

## Estimate local projections

Local projections estimate a separate horizon-specific regression. The IV variant instruments the shock variable and should use an application-specific instrument with a defensible exclusion restriction.

In [7]:
lp_model = LocalProjectionsModel(target="output_growth", shock_var="inflation", horizon=8)
lp_model.fit(data)
lp_model.compute_irf()

,horizon,effect,stderr,lower_ci,upper_ci,pvalue
0,0,0.373683,0.093922,0.189598,0.557767,0.000069
1,1,0.362519,0.087797,0.190441,0.534598,0.000036
2,2,0.096775,0.105110,-0.109238,0.302787,0.357210
3,3,0.094850,0.098453,-0.098114,0.287814,0.335343
4,4,0.011390,0.100154,-0.184908,0.207688,0.909456
5,5,-0.103172,0.103583,-0.306192,0.099848,0.319236
6,6,-0.200783,0.085961,-0.369263,-0.032303,0.019504
7,7,-0.292414,0.092797,-0.474293,-0.110535,0.001627
8,8,-0.250445,0.091735,-0.430243,-0.070648,0.006332


In [8]:
lp_iv_model = LocalProjectionsIVModel(target_variable="output_growth", shock_variable="inflation", instrument_variable="instrument", horizons=8, date_column="date")
lp_iv_model.fit(data)
lp_iv_model.irf_coefficients

[98.73092198715696,
 37.13530301903038,
 28.17032599966537,
 19.750535679816235,
 -19.758569805047514,
 -8.583727518246212,
 -2.712264227799322,
 -2.505897263628674,
 -1.616821855179735]

## Decompose VAR dynamics

The decomposition utility uses the fitted structural representation. Here it is supplied with the Blanchard--Quah impact matrix from the earlier cell.

In [9]:
decomposition = TimeSeriesDecompositions(bq_model.var_result, B_0=bq_model.B_0)
results = decomposition.run(steps=12)
results["fevd"].shape

(12, 2, 2)

## Next steps

Replace the synthetic data with a documented empirical dataset, preserve the full transformation pipeline, and state the identifying assumptions before interpreting impulse responses. For MATLAB parity, use the bounded Blanchard--Quah comparator described in `docs/validation/matlab_comparator.md`.